# 🎨 محوّل الصور الفني - اليوم الوطني
# AI Style Transfer - National Day

---

## مرحباً بكم في تجربة التحويل الفني! 🇱🇾

### كيفية الاستخدام | How to Use:

1. **ارفع صورتك** أو التقط صورة بالكاميرا | Upload your photo or take a webcam shot
2. **اختر النمط الفني** المفضل لديك | Choose your preferred artistic style
3. **اضغط على زر التحويل** وانتظر النتيجة | Click Transform and wait for the result
4. **حمّل الصورة** الفنية الجديدة | Download your new artistic image

### الأنماط المتاحة | Available Styles:
- 🌌 **أنمي / كرتون** | Anime / Cartoon
- 🎨 **لوحة مائية** | Watercolor Painting
- 🖼️ **لوحة زيتية** | Oil Painting
- ✏️ **رسم بالرصاص** | Pencil Sketch
- 🇱🇾 **النمط الوطني** | National Day Theme

---
> **ملاحظة:** قد تستغرق عملية التحويل من 10 إلى 30 ثانية حسب النمط المختار
> **Note:** Processing may take 10-30 seconds depending on the chosen style

In [ ]:
# Cell 2: Install Dependencies

import sys, torch
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU:  {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('  No GPU -- Runtime > Change runtime type > T4 GPU')

# Install only what Colab is missing (do NOT touch torch/opencv/numpy)
!pip install -q 'diffusers>=0.28' 'transformers>=4.41' 'accelerate>=0.29' safetensors
!pip install -q 'gradio>=4.20'
!pip install -q scipy
print('All packages ready!')


In [ ]:
# Cell 3: Import Libraries & Setup

import torch
import numpy as np
import cv2
import gradio as gr
from PIL import Image, ImageEnhance
from diffusers import StableDiffusionImg2ImgPipeline, DPMSolverMultistepScheduler
import warnings, gc, os, time, base64
from io import BytesIO

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32

print('=' * 55)
print('  AI Style Transfer — Libya Tech & IT Day')
print('  محول الصور بالذكاء الاصطناعي — يوم التقنية')
print('=' * 55)
print(f'Device : {DEVICE.upper()}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'NumPy  : {np.__version__}')
print(f'OpenCV : {cv2.__version__}')

_sd_pipeline = None
print('Libraries loaded — models load on first use.')


In [ ]:
# Cell 4: Load AI Model

def load_stable_diffusion():
    global _sd_pipeline
    if _sd_pipeline is not None:
        return _sd_pipeline

    print('Loading Stable Diffusion v1.5 (~4 GB first run)...')
    try:
        _sd_pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(
            'runwayml/stable-diffusion-v1-5',
            torch_dtype=DTYPE,
            safety_checker=None,
            requires_safety_checker=False,
            token=False,
        )
        _sd_pipeline.scheduler = DPMSolverMultistepScheduler.from_config(
            _sd_pipeline.scheduler.config
        )
        _sd_pipeline = _sd_pipeline.to(DEVICE)
        if DEVICE == 'cuda':
            _sd_pipeline.enable_attention_slicing()
            _sd_pipeline.enable_vae_slicing()
            try:
                _sd_pipeline.enable_xformers_memory_efficient_attention()
                print('  xFormers enabled')
            except Exception:
                pass
        print('Model ready!')
        if DEVICE == 'cuda':
            used  = torch.cuda.memory_allocated()/1e9
            total = torch.cuda.get_device_properties(0).total_memory/1e9
            print(f'  VRAM used: {used:.1f} / {total:.1f} GB')
        return _sd_pipeline
    except Exception as e:
        print(f'Could not load SD model: {e}')
        print('OpenCV styles still work without it.')
        return None

load_stable_diffusion()


In [ ]:
# Cell 5: Style Transfer Functions

# ── Helpers ──────────────────────────────────────────────

def pil_to_cv2(img):
    return cv2.cvtColor(np.array(img.convert('RGB')), cv2.COLOR_RGB2BGR)

def cv2_to_pil(img):
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

def resize_keep_aspect(img, max_side=768):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    s = max_side / max(w, h)
    nw, nh = int(w*s)//8*8, int(h*s)//8*8
    return img.resize((max(nw,64), max(nh,64)), Image.LANCZOS)

def resize_to_sd(img, target=512):
    w, h = img.size
    if w >= h:
        nw, nh = target, int(h*target/w)
    else:
        nw, nh = int(w*target/h), target
    return img.resize((max(nw//8*8,64), max(nh//8*8,64)), Image.LANCZOS)


# ── 1. Anime (Stable Diffusion) ──────────────────────────

def anime_style(image, strength=0.55):
    pipe = load_stable_diffusion()
    if pipe is None:
        return cartoon_cv(image)
    img = resize_to_sd(image.convert('RGB'), 512)
    prompt = ('anime style, masterpiece, best quality, detailed, '
              'vibrant colors, studio ghibli, clean lines, beautiful illustration')
    neg    = ('realistic, photo, blurry, low quality, ugly, deformed')
    try:
        with torch.autocast(DEVICE if DEVICE=='cuda' else 'cpu'):
            out = pipe(prompt=prompt, negative_prompt=neg, image=img,
                       strength=strength, guidance_scale=7.5,
                       num_inference_steps=25).images[0]
        if DEVICE == 'cuda': torch.cuda.empty_cache(); gc.collect()
        return out.resize(image.size, Image.LANCZOS)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect()
        out = pipe(prompt=prompt, negative_prompt=neg, image=img,
                   strength=strength, guidance_scale=7.0,
                   num_inference_steps=20).images[0]
        torch.cuda.empty_cache()
        return out.resize(image.size, Image.LANCZOS)

def cartoon_cv(image):
    img = pil_to_cv2(image)
    img = cv2.resize(img, (512,512))
    for _ in range(3): img = cv2.bilateralFilter(img, 9, 75, 75)
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.adaptiveThreshold(cv2.medianBlur(gray,7), 255,
                cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 2)
    edges = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    data  = np.float32(img).reshape((-1,3))
    crit  = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 20, 0.001)
    _, lbl, ctr = cv2.kmeans(data, 12, None, crit, 10, cv2.KMEANS_RANDOM_CENTERS)
    quant = np.uint8(ctr)[lbl.flatten()].reshape(img.shape)
    return cv2_to_pil(cv2.bitwise_and(quant, edges))


# ── 2. Watercolor (OpenCV) ────────────────────────────────

def watercolor_style(image, strength=0.6):
    img = pil_to_cv2(image)
    h, w = img.shape[:2]
    if max(h,w) > 1024:
        s = 1024/max(h,w); img = cv2.resize(img,(int(w*s),int(h*s)))
    out = cv2.stylization(img, sigma_s=int(60+strength*60),
                          sigma_r=0.3+strength*0.25)
    out = cv2.addWeighted(out, 0.7, cv2.GaussianBlur(out,(3,3),0), 0.3, 0)
    hsv = cv2.cvtColor(out, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:,:,1] = np.clip(hsv[:,:,1]*1.3, 0, 255)
    out = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    return ImageEnhance.Color(cv2_to_pil(out)).enhance(1.15)


# ── 3. Oil Painting (OpenCV) ──────────────────────────────

def oil_painting_style(image, strength=0.6):
    img = pil_to_cv2(image)
    h, w = img.shape[:2]
    if max(h,w) > 1024:
        s = 1024/max(h,w); img = cv2.resize(img,(int(w*s),int(h*s)))
    out  = cv2.detailEnhance(img, sigma_s=int(10+strength*10), sigma_r=0.15)
    n    = max(6, int(16-strength*8))
    data = np.float32(out).reshape((-1,3))
    crit = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 20, 0.5)
    _, lbl, ctr = cv2.kmeans(data, n, None, crit, 5, cv2.KMEANS_RANDOM_CENTERS)
    quant = np.uint8(ctr)[lbl.flatten()].reshape(out.shape)
    blend = cv2.edgePreservingFilter(
        cv2.addWeighted(out,0.4,quant,0.6,0), flags=1, sigma_s=30, sigma_r=0.4)
    res = ImageEnhance.Contrast(cv2_to_pil(blend)).enhance(1.1)
    return ImageEnhance.Color(res).enhance(1.2)


# ── 4. Pencil Sketch (OpenCV) ─────────────────────────────

def pencil_sketch(image, strength=0.6):
    img = pil_to_cv2(image)
    h, w = img.shape[:2]
    if max(h,w) > 1024:
        s = 1024/max(h,w); img = cv2.resize(img,(int(w*s),int(h*s)))
    gray, _ = cv2.pencilSketch(img, sigma_s=60, sigma_r=0.07,
                                shade_factor=0.03+strength*0.05)
    res = cv2_to_pil(cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR))
    return ImageEnhance.Contrast(res).enhance(1.3)


# ── 5. Tech Day (blue digital overlay) ───────────────────

def tech_day_style(image, strength=0.55):
    result = watercolor_style(image, strength=strength * 0.8)
    arr  = np.array(result).astype(np.float32)
    # Blue digital tint (Libya Tech Day)
    mask = np.zeros_like(arr)
    mask[:,:,2] = 200.0   # R
    mask[:,:,1] = 30.0    # G
    mask[:,:,0] = 0.0     # B
    tinted = np.clip(arr*0.85 + mask*0.15, 0, 255).astype(np.uint8)
    return ImageEnhance.Color(Image.fromarray(tinted)).enhance(1.25)


# ── Style map & dispatcher ────────────────────────────────

STYLE_MAP = {
    'Anime / انمي':               ('anime',   anime_style),
    'Watercolor / الوان مائية':   ('water',   watercolor_style),
    'Oil Painting / زيتية':       ('oil',     oil_painting_style),
    'Pencil Sketch / رصاص':       ('sketch',  pencil_sketch),
    'Tech Day / يوم التقنية':     ('tech',    tech_day_style),
}


def transform_image(image, style_name, strength=0.55):
    if image is None:
        return None, 'Please upload an image first | ارفع صورة اولا'
    try:
        pil = Image.fromarray(image).convert('RGB') if isinstance(image, np.ndarray) \
              else image.convert('RGB')
        pil = resize_keep_aspect(pil, 768)
        _, fn = STYLE_MAP.get(style_name, ('anime', anime_style))
        t0     = time.time()
        result = fn(pil, strength=strength)
        elapsed = time.time() - t0
        status = (f'Done in {elapsed:.1f}s  |  Style: {style_name}  |  '
                  f'Size: {result.width}x{result.height}px')
        return result, status
    except Exception as e:
        err = str(e)
        if 'out of memory' in err.lower():
            return None, 'GPU out of memory — try smaller image or restart runtime'
        return None, f'Error: {err[:200]}'


print('Style functions ready. Available styles:')
for k in STYLE_MAP: print(f'  {k}')


In [ ]:
# Cell 6: Gradio UI

import gradio as gr

STYLES = list(STYLE_MAP.keys())

with gr.Blocks(
    title='AI Style Transfer — Libya Tech & IT Day',
    theme=gr.themes.Soft(primary_hue='blue'),
) as demo:

    gr.Markdown('# AI Style Transfer')
    gr.Markdown('## محول الصور بالذكاء الاصطناعي — يوم التقنية والمعلومات - ليبيا')
    gr.Markdown('حول صورتك الى لوحة فنية في ثوان | Transform your photo into art in seconds')

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(
                label='Your Photo | صورتك',
                type='pil',
                sources=['upload', 'webcam', 'clipboard'],
                height=320,
            )
            style_selector = gr.Radio(
                choices=STYLES,
                value=STYLES[0],
                label='Art Style | النمط الفني',
            )
            strength_slider = gr.Slider(
                minimum=0.30, maximum=0.80, value=0.55, step=0.05,
                label='Style Strength (Anime only) | قوة التاثير للانمي فقط',
            )
            transform_btn = gr.Button('Transform | حول الان', variant='primary')

        with gr.Column(scale=1):
            output_image = gr.Image(
                label='Result | النتيجة',
                type='pil',
                height=320,
                interactive=False,
            )
            status_box = gr.Textbox(
                label='Status | الحالة',
                lines=2,
                interactive=False,
            )
            download_btn = gr.DownloadButton(
                label='Download | تحميل',
                visible=False,
            )

    gr.Markdown(
        'Speed: Sketch / Watercolor / Oil / Tech Day = instant  |  '
        'Anime = 15-30s on GPU'
    )

    def on_transform(image, style, strength):
        result, status = transform_image(image, style, strength)
        if result is not None:
            tmp = '/tmp/styled_image.png'
            result.save(tmp)
            return result, status, gr.update(value=tmp, visible=True)
        return None, status, gr.update(visible=False)

    transform_btn.click(
        fn=on_transform,
        inputs=[input_image, style_selector, strength_slider],
        outputs=[output_image, status_box, download_btn],
    )

print('Launching...')
demo.queue(max_size=5).launch(share=True, debug=False, show_error=True)


In [ ]:
# ============================================================
# Cell 7: Wireless Printing | Tabia Lasilkiya
# ============================================================
# Adds a branded National Day frame and opens the browser print
# dialog so the image is sent to any connected WiFi printer.
# ============================================================

import base64, io, datetime
from PIL import Image, ImageDraw, ImageFont
import gradio as gr


# -- A: Branded print card ---------------------------------

def create_print_card(
    image,
    title_ar='يوم التقنية والمعلومات - ليبيا',
    title_en='Libya Tech & IT Day',
    subtitle='AI Style Transfer . تحويل الصور بالذكاء الاصطناعي',
    card_size=(1200, 900),
):
    """Wrap the styled image in a branded National Day print card."""
    W, H    = card_size
    BORDER  = 30
    HEADER  = 80
    FOOTER  = 55
    PADDING = 12
    GREEN   = (10, 22, 40)
    GOLD    = (0, 168, 255)

    canvas = Image.new('RGB', (W, H), color=GREEN)

    ix0, iy0 = BORDER, BORDER + HEADER
    ix1, iy1 = W - BORDER, H - BORDER - FOOTER
    canvas.paste(Image.new('RGB', (ix1-ix0, iy1-iy0), 'white'), (ix0, iy0))

    inner_w = ix1 - ix0 - 2*PADDING
    inner_h = iy1 - iy0 - 2*PADDING
    thumb = image.copy().convert('RGB')
    thumb.thumbnail((inner_w, inner_h), Image.LANCZOS)
    px = ix0 + PADDING + (inner_w - thumb.width)  // 2
    py = iy0 + PADDING + (inner_h - thumb.height) // 2
    canvas.paste(thumb, (px, py))

    draw = ImageDraw.Draw(canvas)

    draw.rectangle([BORDER//2, BORDER//2, W-BORDER//2, H-BORDER//2],
                   outline=GOLD, width=3)
    draw.rectangle([BORDER//2+6, BORDER//2+6, W-BORDER//2-6, H-BORDER//2-6],
                   outline=GOLD, width=1)

    def font(px):
        for path in [
            '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
            '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
            '/usr/share/fonts/truetype/freefont/FreeSansBold.ttf',
        ]:
            try:
                return ImageFont.truetype(path, px)
            except Exception:
                pass
        return ImageFont.load_default()

    draw.text((W//2, BORDER + HEADER//2),
              f'{title_ar}  |  {title_en}',
              fill=GOLD, font=font(30), anchor='mm')

    draw.text((W//2, H - BORDER - FOOTER//2 - 10),
              subtitle,
              fill=(180, 220, 180), font=font(19), anchor='mm')
    draw.text((W//2, H - BORDER - FOOTER//2 + 14),
              datetime.datetime.now().strftime('%Y/%m/%d'),
              fill=(150, 190, 160), font=font(16), anchor='mm')

    sq = 14
    for cx, cy in [(BORDER+4, BORDER+4), (W-BORDER-4-sq, BORDER+4),
                   (BORDER+4, H-BORDER-4-sq), (W-BORDER-4-sq, H-BORDER-4-sq)]:
        draw.rectangle([cx, cy, cx+sq, cy+sq], fill=GOLD)

    return canvas


# -- B: Auto-print HTML builder ---------------------------

def build_print_html(image):
    """Encode image as base64 and return HTML with auto-print JavaScript."""
    if image is None:
        return "<p style='color:#f88;'>Upload image first</p>"

    buf = io.BytesIO()
    image.save(buf, format='PNG', dpi=(300, 300))
    b64 = base64.b64encode(buf.getvalue()).decode()

    html_page = (
        '<!DOCTYPE html><html><head><title>National Day Print</title>'
        '<style>'
        'body{margin:0;padding:0;background:#fff;display:flex;'
        'justify-content:center;align-items:center;min-height:100vh;}'
        'img{max-width:100%;max-height:97vh;object-fit:contain;}'
        '@media print{img{width:100%;height:auto;page-break-inside:avoid;}}'
        '</style></head><body>'
        f'<img src="data:image/png;base64,{b64}"'
        ' onload="setTimeout(()=>{window.print();setTimeout(()=>window.close(),2500);},400)">'
        '</body></html>'
    )

    # Escape for JavaScript string
    js_page = html_page.replace('`', r'\`').replace('${', r'\${')

    return (
        '<div style="text-align:center;padding:14px;">'
        '<button onclick="openPrint()" style="'
        'background:linear-gradient(135deg,#006C35,#00A550);'
        'color:white;border:2px solid #C8A951;'
        'padding:14px 38px;font-size:1.15em;font-weight:bold;'
        'border-radius:10px;cursor:pointer;">'
        '&#128424; &nbsp; '
        '\u0637\u0628\u0627\u0639\u0629 \u0639\u0644\u0649 \u0627\u0644\u0637\u0627\u0628\u0639\u0629 \u0627\u0644\u0644\u0627\u0633\u0644\u0643\u064a\u0629'
        ' &nbsp;&middot;&nbsp; Print to Wireless Printer'
        '</button>'
        '<p style="color:#aaa;font-size:0.82em;margin:6px 0 0;">'
        '\u0633\u062a\u0641\u062a\u062d \u0646\u0627\u0641\u0630\u0629 \u0627\u0644\u0637\u0627\u0628\u0639\u0629 \u062a\u0644\u0642\u0627\u0626\u064a\u0627\u064b'
        ' &nbsp;&middot;&nbsp; Print dialog opens automatically'
        '</p></div>'
        f'<script>function openPrint(){{var w=window.open("","_blank","width=980,height=740");w.document.write(`{js_page}`);w.document.close();}}\n</script>'
    )


# -- C: Printing UI ---------------------------------------

print_css = 'body, .gradio-container { background: linear-gradient(135deg, #0A2B1A, #0D3B22) !important; }'

with gr.Blocks(css=print_css, title='Print | Tiba3a') as print_demo:

    gr.HTML("""
    <div style='text-align:center;padding:20px 0 8px;'>
      <span style='color:#C8A951;font-size:1.7em;font-weight:bold;'>
        &#128424; \u0637\u0628\u0627\u0639\u0629 \u0627\u0644\u0635\u0648\u0631\u0629 \u0627\u0644\u0641\u0646\u064a\u0629 &nbsp;&middot;&nbsp; Print Your Art
      </span><br>
      <span style='color:#8FD4A8;font-size:0.95em;'>
        \u0627\u0644\u064a\u0648\u0645 \u0627\u0644\u0648\u0637\u0646\u064a \u0627\u0644\u0633\u0639\u0648\u062f\u064a &#127480;&#127462;
      </span>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            print_input = gr.Image(
                label='Image to Print',
                type='pil',
                sources=['upload', 'clipboard'],
                height=310,
            )
            add_frame = gr.Checkbox(
                label='Add National Day Frame (green + gold)',
                value=True,
            )
            print_btn = gr.Button(
                '&#128424; Print Now',
                variant='primary',
            )

        with gr.Column(scale=1):
            preview = gr.Image(
                label='Print Preview',
                type='pil',
                height=310,
                interactive=False,
            )
            print_html = gr.HTML()

    def on_print(img, frame):
        if img is None:
            return None, "<p style='color:#f88;text-align:center;'>Upload image first</p>"
        card = create_print_card(img) if frame else img
        return card, build_print_html(card)

    print_btn.click(
        fn=on_print,
        inputs=[print_input, add_frame],
        outputs=[preview, print_html],
    )

    gr.HTML("""
    <div style='background:rgba(200,169,81,0.08);border:1px solid rgba(200,169,81,0.3);
                border-radius:10px;padding:14px;margin-top:12px;
                color:#C8A951;font-size:0.88em;line-height:1.8;'>
      <strong>Steps / \u062e\u0637\u0648\u0627\u062a:</strong><br>
      1. Upload the styled image (or paste from clipboard)<br>
      2. Enable 'National Day Frame' for green+gold border<br>
      3. Click Print Now -- print dialog opens automatically<br>
      4. Select your wireless printer and print!<br><br>
      <strong>Tip:</strong> Printer must be on same WiFi with driver installed
    </div>
    """)

print('=' * 60)
print('Wireless Printing Interface ready!')
print('=' * 60)
print_demo.launch(share=True, quiet=True)


# 💡 Tips & Troubleshooting | نصائح واستكشاف الأخطاء

---

## ⚡ Performance Tips | نصائح الأداء

### Getting the Best Results | للحصول على أفضل النتائج:

| Style | Best Input | Strength | Time |
|-------|-----------|----------|------|
| 🌌 Anime | Portrait, face photo | 0.5-0.65 | ~25s |
| 🎨 Watercolor | Landscape, colorful | 0.5-0.7 | ~3s |
| 🖼️ Oil Painting | Portrait, any photo | 0.5-0.7 | ~5s |
| ✏️ Pencil Sketch | Any photo | 0.3-0.6 | ~2s |
| 🇱🇾 National Day | Portrait, group | 0.5-0.65 | ~5s |

---

## 🔧 Common Issues | المشكلات الشائعة

### ❌ "GPU out of memory" Error
```
Solution:
1. Runtime > Restart runtime
2. Re-run all cells
3. Use a smaller image (< 512px)
4. Reduce Style Strength slider
```

### ❌ Anime style is slow or fails
```
Possible causes:
• Model still downloading (first run takes 2-5 min)
• GPU memory full → restart runtime
• No GPU selected → Runtime > Change runtime type > T4

Solutions:
• Wait for download to complete
• Use watercolor/sketch while waiting
• Restart and re-run cells
```

### ❌ "No GPU detected" warning
```
Steps to enable GPU:
1. Runtime → Change runtime type
2. Hardware accelerator → T4 GPU
3. Save → Disconnect and Reconnect
4. Re-run all cells from top
```

### ❌ Gradio link expired
```
Free Gradio share links expire after 72 hours.
Re-run Cell 6 to get a new link.
```

---

## 📱 For Event Staff | لموظفي الفعالية

### Setup Checklist:
- [ ] Open notebook in Google Colab
- [ ] Enable T4 GPU (Runtime > Change runtime type)
- [ ] Run all cells in order (Runtime > Run all)
- [ ] Wait for the public Gradio URL to appear
- [ ] Share the URL or QR code with visitors
- [ ] Keep the Colab tab open (do not close!)

### Visitor Instructions (Arabic):
```
مرحباً بك في تجربة التحويل الفني! 🎨
1. ارفع صورتك أو اسحبها
2. اختر النمط الفني المفضل
3. اضغط "حوّل الآن"
4. انتظر 5-30 ثانية
5. حمّل الصورة الفنية الجديدة!
```

### Quick Reset if Something Goes Wrong:
1. Runtime → Restart and Run All
2. Wait ~5 minutes for models to reload
3. Share the new Gradio URL with visitors

---

## 🌟 Advanced Usage | الاستخدام المتقدم

### Customizing Prompts (Anime Style):
To modify the anime style prompt, edit Cell 5, `anime_style()` function:
```python
prompt = "YOUR CUSTOM PROMPT, anime style, ..."
negative_prompt = "realistic, photo, blurry, ..."
```

### Adding New Styles:
1. Add a new function in Cell 5 following the same pattern
2. Add it to the `STYLE_MAP` dictionary
3. Re-run Cell 5 and Cell 6

### Batch Processing:
For multiple images, call `transform_image()` directly:
```python
from PIL import Image
img = Image.open("your_photo.jpg")
result, status = transform_image(img, "🌌 Anime / أنمي", strength=0.6)
result.save("output.png")
```

---

## 📞 Technical Specs | المواصفات التقنية

- **Base Model**: Stable Diffusion v1.5 (runwayml/stable-diffusion-v1-5)
- **CV Styles**: OpenCV 4.8+ (watercolor, oil, sketch, national day)
- **GPU**: Google Colab T4 (16 GB VRAM)
- **Framework**: Gradio 3.50 + Diffusers 0.21
- **Target**: < 30 seconds per image
- **Max image size**: 768×768 px (auto-resized)

---

*🇱🇾 Developed for Libya Tech & IT Day Event · تطوير لفعالية يوم التقنية والمعلومات - ليبيا*